# Quick Start: Five Lines to Structured Data

Discover a schema, extract records, and normalize values — all in five lines of Python. No need to specify which fields to normalize — catchfly figures that out automatically.

**What you'll learn:** How `Pipeline.quick()` automates the entire discovery → extraction → field selection → normalization pipeline.

**Estimated API cost:** ~$0.01

In [ ]:
# pip install "catchfly[openai,export]"
# export OPENAI_API_KEY="sk-..."

## The five lines

In [ ]:
from catchfly import Pipeline
from catchfly.demo import load_samples

docs = load_samples("product_reviews")

pipeline = Pipeline.quick(model="gpt-5.4-mini")

results = pipeline.run(docs, domain_hint="Electronics product reviews")

In [ ]:
df = results.to_dataframe()
df[["product_name", "brand", "category", "rating", "price", "pros", "cons"]].head(10)

That's it. Catchfly discovered a schema from your documents, extracted structured records, **automatically identified which fields are categorical**, and normalized them — grouping variants like "great battery" and "long battery life" into a single canonical form.

No `normalize_fields` parameter needed. The built-in `StatisticalFieldSelector` analyzed the extracted values and picked the right fields.

## What happened under the hood

### 1. Schema discovery

Catchfly sampled your documents and asked the LLM to infer a schema:

In [ ]:
import json

print(json.dumps(results.schema.json_schema, indent=2))

### 2. Extracted records

Each document was sent to the LLM with the discovered schema. The result: typed Pydantic model instances with provenance tracking.

In [ ]:
for record in results.records[:5]:
    print(f"{record.product_name}: pros={record.pros}")

### 3. Field selection

The `StatisticalFieldSelector` analyzed each string field's values — cardinality, average length, name patterns — and decided which ones are categorical (worth normalizing) vs free-text (skip):



In [ ]:
# Which fields were auto-selected for normalization?
print("Normalized fields:", list(results.normalizations.keys()))
print()

# Inspect the clusters for each normalized field
for field_name, norm in results.normalizations.items():
    print(f"'{field_name}' — {len(norm.clusters)} canonical groups:")
    for canonical, members in norm.clusters.items():
        if len(members) > 1:
            print(f"  {canonical}: {members}")
    print()

### 4. Overriding field selection

You can always bypass auto-selection with an explicit list:

```python
results = pipeline.run(docs, normalize_fields=["pros"])  # only normalize "pros"
results = pipeline.run(docs, normalize_fields="all")     # normalize all string fields
```

### 5. Cost report

In [ ]:
print(f"Total cost: ${results.report.total_cost_usd:.4f}")
print(f"Total tokens: {results.report.total_input_tokens + results.report.total_output_tokens:,}")

In [ ]:
estimate = pipeline.estimate_cost(docs)
print(estimate)

In [ ]:
pipeline = Pipeline.quick(model="gpt-5.4-mini")
estimate = pipeline.estimate_cost(docs)
print(estimate)

## Next steps

- [02_rare_disease.ipynb](02_rare_disease.ipynb) — Full pipeline with HPO ontology coding
- [03_product_catalog.ipynb](03_product_catalog.ipynb) — Cost control, checkpoints, per-field normalization
- [04_custom_schema.ipynb](04_custom_schema.ipynb) — Skip discovery, use your own Pydantic models
- [05_local_models.ipynb](05_local_models.ipynb) — Run everything locally with Ollama